# ETL e Ingesta a SQL Server - Predicción de Incendios Forestales
Este cuaderno contiene el proceso de Extracción, Transformación y Carga (ETL) para poblar la base de datos relacional normalizada en **SQL Server** utilizando los datasets oficiales de la NASA (FIRMS, POWER y MODIS).

## 1. Importación de Librerías y Configuración

In [1]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import time

# Configuración de conexión al servidor SQL Server (Instancia LUIS)
CONN_STR = 'mssql+pyodbc://LUIS/IncendiosForestalesEC?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes'
engine = create_engine(CONN_STR)
print('Motor de SQLAlchemy configurado correctamente.')

Motor de SQLAlchemy configurado correctamente.


## 2. Inicialización y Limpieza de Tablas (Truncate)
Limpiamos las tablas en orden de dependencias de llaves foráneas para evitar conflictos en recargas.

In [2]:
with engine.connect() as conn:
    print('Limpiando tablas existentes...')
    conn.execute(text('DELETE FROM Incendios;'))
    conn.execute(text('DELETE FROM Clima;'))
    conn.execute(text('DELETE FROM NDVI;'))
    conn.execute(text('DELETE FROM Ciudades;'))
    conn.commit()
    print('Tablas limpiadas con éxito.')

Limpiando tablas existentes...


C:\Users\gonza\AppData\Local\Temp\ipykernel_26148\4211611196.py:1: SAWarning: Unrecognized server version info '17.0.1115.1'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


Tablas limpiadas con éxito.


## 3. Cargar e Insertar Ciudades (Dimensión)
Insertamos las ciudades bajo estudio con sus respectivas coordenadas y altitud msnm.

In [3]:
cities_data = [
    {'id_ciudad': 1, 'nombre': 'Quito', 'region': 'Sierra', 'latitud': -0.1807, 'longitud': -78.4678, 'altitud_msnm': 2850.0},
    {'id_ciudad': 2, 'nombre': 'Guayaquil', 'region': 'Costa', 'latitud': -2.1894, 'longitud': -79.8890, 'altitud_msnm': 4.0},
    {'id_ciudad': 3, 'nombre': 'Riobamba', 'region': 'Sierra', 'latitud': -1.6731, 'longitud': -78.6530, 'altitud_msnm': 2754.0},
    {'id_ciudad': 4, 'nombre': 'Cuenca', 'region': 'Sierra', 'latitud': -2.9001, 'longitud': -79.0060, 'altitud_msnm': 2560.0}
]
df_cities = pd.DataFrame(cities_data)
df_cities.to_sql('Ciudades', con=engine, if_exists='append', index=False)
print('Ciudades registradas en SQL Server.')

Ciudades registradas en SQL Server.


## 4. Transformación y Carga de Clima Diario (NASA POWER)
Cargamos los datos climáticos diarios. Reemplazamos los valores nulos mediante interpolación agrupada por ciudad para satisfacer la restricción `NOT NULL`.

In [4]:
df_clima = pd.read_csv('clima_diario_4ciudades.csv')

# Imputar valores nulos con interpolación de días adyacentes por ciudad
df_clima = df_clima.groupby('ciudad').apply(lambda g: g.ffill().bfill()).reset_index(drop=True)

# Mapear nombres de ciudades a sus IDs correspondientes
city_map = {'Quito': 1, 'Guayaquil': 2, 'Riobamba': 3, 'Cuenca': 4}
df_clima['id_ciudad'] = df_clima['ciudad'].map(city_map)

# Seleccionar y ordenar columnas según el esquema de base de datos
clima_cols = ['id_ciudad', 'fecha', 'velocidad_viento', 'direccion_viento', 
              'temperatura_media', 'temperatura_max', 'temperatura_min', 
              'humedad_relativa', 'precipitacion']
df_clima_final = df_clima[clima_cols].drop_duplicates(subset=['id_ciudad', 'fecha'])

print(f'Ingresando {len(df_clima_final)} registros de Clima...')
df_clima_final.to_sql('Clima', con=engine, if_exists='append', index=False, chunksize=2000)
print('Datos climáticos guardados.')

Ingresando 21100 registros de Clima...


C:\Users\gonza\AppData\Local\Temp\ipykernel_26148\926384240.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_clima = df_clima.groupby('ciudad').apply(lambda g: g.ffill().bfill()).reset_index(drop=True)


Datos climáticos guardados.


## 5. Transformación y Carga de Vegetación NDVI (MODIS MOD13Q1)
Importamos el dataset de NDVI, mapeando la columna `ID` al identificador de la ciudad.

In [5]:
df_ndvi = pd.read_csv('NDVI-Ecuador-Cities-MOD13Q1-061-results.csv')

# Mapeo de ciudades
df_ndvi['id_ciudad'] = df_ndvi['ID'].map(city_map)
df_ndvi = df_ndvi[df_ndvi['id_ciudad'].notnull()]
df_ndvi['id_ciudad'] = df_ndvi['id_ciudad'].astype(int)

# Renombrar y completar columnas
df_ndvi = df_ndvi.rename(columns={
    'Date': 'fecha',
    'MOD13Q1_061__250m_16_days_NDVI': 'ndvi',
    'MOD13Q1_061__250m_16_days_VI_Quality': 'vi_quality',
    'MOD13Q1_061__250m_16_days_pixel_reliability': 'pixel_reliability',
    'MODIS_Tile': 'modis_tile'
})
df_ndvi['evi'] = None  # Columna nullable en esquema

# Limpiar nulos
df_ndvi['ndvi'] = df_ndvi['ndvi'].ffill().bfill()
df_ndvi['vi_quality'] = df_ndvi['vi_quality'].fillna(0).astype(int)
df_ndvi['pixel_reliability'] = df_ndvi['pixel_reliability'].fillna(0).astype(int)
df_ndvi['modis_tile'] = df_ndvi['modis_tile'].fillna('h10v09')

ndvi_cols = ['id_ciudad', 'fecha', 'ndvi', 'evi', 'vi_quality', 'pixel_reliability', 'modis_tile']
df_ndvi_final = df_ndvi[ndvi_cols].drop_duplicates(subset=['id_ciudad', 'fecha'])

print(f'Ingresando {len(df_ndvi_final)} registros en NDVI...')
df_ndvi_final.to_sql('NDVI', con=engine, if_exists='append', index=False)
print('Datos NDVI guardados.')

Ingresando 1328 registros en NDVI...
Datos NDVI guardados.


## 6. Mapeo Vectorial e Ingesta de Incendios Históricos (NASA FIRMS)
Para asociar los focos de calor a nivel nacional con las ciudades bajo estudio, calculamos la distancia euclidiana de forma vectorial optimizada en NumPy.

In [6]:
df_incendios = pd.read_csv('fire_archive_M-C61_761089.csv')

# Coordenadas de las ciudades para mapeo espacial
cities_coords = {
    1: (-0.1807, -78.4678),  # Quito
    2: (-2.1894, -79.8890),  # Guayaquil
    3: (-1.6731, -78.6530),  # Riobamba
    4: (-2.9001, -79.0060)   # Cuenca
}

lats = df_incendios['latitude'].values
lons = df_incendios['longitude'].values

dists = []
city_ids = sorted(cities_coords.keys())
for cid in city_ids:
    clat, clon = cities_coords[cid]
    dist = np.sqrt((lats - clat)**2 + (lons - clon)**2)
    dists.append(dist)
    
dists = np.column_stack(dists)
closest_indices = np.argmin(dists, axis=1)
df_incendios['id_ciudad'] = [city_ids[idx] for idx in closest_indices]

# Mapear y formatear columnas según el esquema relacional
df_incendios_final = df_incendios.rename(columns={
    'latitude': 'latitud',
    'longitude': 'longitud',
    'acq_date': 'fecha_deteccion',
    'acq_time': 'hora_deteccion',
    'satellite': 'satelite',
    'instrument': 'instrumento',
    'confidence': 'confianza',
    'daynight': 'dia_noche',
    'type': 'tipo'
})

df_incendios_final['dia_noche'] = df_incendios_final['dia_noche'].str.strip().str[0]

incendios_cols = ['id_ciudad', 'latitud', 'longitud', 'brightness', 'scan', 'track', 
                  'fecha_deteccion', 'hora_deteccion', 'satelite', 'instrumento', 
                  'confianza', 'version', 'bright_t31', 'frp', 'dia_noche', 'tipo']
df_incendios_final = df_incendios_final[incendios_cols]

print(f'Insertando {len(df_incendios_final)} registros en Incendios...')
t0 = time.time()
df_incendios_final.to_sql('Incendios', con=engine, if_exists='append', index=False, chunksize=5000)
print(f'Incendios insertados con éxito en {time.time() - t0:.2f} segundos.')

Insertando 72532 registros en Incendios...
Incendios insertados con éxito en 3.72 segundos.


## 7. Verificación de Ingesta y Auditoría
Realizamos un conteo final y verificamos que los triggers de auditoría se hayan ejecutado.

In [7]:
with engine.connect() as conn:
    r_ciu = conn.execute(text('SELECT COUNT(*) FROM Ciudades;')).fetchone()[0]
    r_cli = conn.execute(text('SELECT COUNT(*) FROM Clima;')).fetchone()[0]
    r_ndv = conn.execute(text('SELECT COUNT(*) FROM NDVI;')).fetchone()[0]
    r_inc = conn.execute(text('SELECT COUNT(*) FROM Incendios;')).fetchone()[0]
    r_aud = conn.execute(text('SELECT COUNT(*) FROM Auditoria;')).fetchone()[0]
    
print('=== VERIFICACIÓN DE INGESTA ===')
print(f'Ciudades en BD: {r_ciu}')
print(f'Registros Clima: {r_cli}')
print(f'Registros NDVI: {r_ndv}')
print(f'Registros Incendios: {r_inc}')
print(f'Registros en Auditoría (Triggers ejecutados): {r_aud}')

=== VERIFICACIÓN DE INGESTA ===
Ciudades en BD: 4
Registros Clima: 21100
Registros NDVI: 1328
Registros Incendios: 72532
Registros en Auditoría (Triggers ejecutados): 1987
